# Steam Text

Steam 게임 추천에서 텍스트 파트만 따로 정리한 notebook입니다.  
이미지는 다른 파트에서 처리하므로 여기서는 `title`, `tags`, `description`으로 게임 1개당 텍스트 임베딩 1개를 만드는 데 집중합니다.

## 1. 텍스트 전처리

`games.csv`와 `games_metadata.json`을 `app_id` 기준으로 합치고, 비교용 텍스트 컬럼을 만듭니다.

In [ ]:
!python steam_text_preprocessing.py

## 2. Token 길이 확인

MiniLM tokenizer 기준으로 `title + tags + description`의 p95는 105 token, p99는 121 token입니다.  
256 token 초과는 21개뿐이라 첫 baseline에서는 chunking 대신 truncation만 둡니다.

In [ ]:
import pandas as pd

pd.read_csv('text_token_length_summary.csv')

## 3. 텍스트 임베딩 생성

`sentence-transformers/all-MiniLM-L6-v2`를 사용합니다. 출력 차원은 384입니다.

In [ ]:
!python 09_encode_text_embeddings.py --batch-size 128 --device cpu

In [ ]:
import numpy as np

emb = np.load('emb_text_minilm.npy')
emb.shape, emb.dtype

## 4. 이웃 점검

정량 평가가 아니라 임베딩과 `app_id` 매핑을 확인하는 기본 점검입니다.

In [ ]:
!python 10_text_neighbors.py

In [ ]:
pd.read_csv('text_neighbors_sample.csv').head(20)

## 5. Text Tower

fusion에서는 384차원 텍스트 임베딩을 바로 concat하지 않고, 이미지 타워와 같은 64차원으로 한 번 사영합니다.

In [ ]:
!python 08_text_tower.py

정리: baseline text는 `title + tags + description`, encoder는 MiniLM, 최종 text embedding은 `(50872, 384)`입니다.  
fusion 단계에서는 `TextTower`를 통해 `(B, 384) -> (B, 64)`로 맞춘 뒤 이미지/정형 feature와 결합합니다.